### Structured Output

Structured output is a powerful feature in LangChain that force the LLM to rutn it's response in a specifi, predicatable format (Like, JSON, dictionary, or Python object) instead of plain free-form text.

### Pydantic
Pydantic is a library used to define data models with automatic validation, parsing, and error handling. 

In [3]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F599277C50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F59940C690>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the Movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the Movie")
    rating:float=Field(description="This is the rating of the movie out of 10")

In [6]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure


_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F599277C50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F59940C690>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the Movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the Movie', 'type': 'string'}, 'rating': {'description': 'This is the rating of

In [7]:
model.invoke("Tell me about movie Inception")


AIMessage(content='<think>\nOkay, so I need to explain the movie Inception. Let me recall what I know. Inception is a 2010 film directed by Christopher Nolan, right? The main actor is Leonardo DiCaprio. The title, "Inception," refers to the act of planting an idea into someone\'s subconscious. The story revolves around a thief who steals information from people\'s dreams, and he\'s offered a chance to erase his criminal past by performing the inverse: planting an idea instead.\n\nThe protagonist is Dom Cobb, played by DiCaprio. He\'s a professional thief who enters people\'s dreams to steal secrets. The term they use is "extraction." The antagonist might be someone like Mal, Cobb\'s wife, who has a tragic background. She was in a wheelchair after a dream accident and couldn\'t distinguish between dreams and reality, leading her to take her own life. That\'s a significant part of Cobb\'s guilt.\n\nThe film uses layers of dreams within dreams. Each level has different time dilation, mean

In [32]:
response=model_with_structure.invoke("Tell me about movie Spiderman 1")
response

Movie(title='Spiderman 1', year=2002, director='Sam Raimi', rating=7.3)

### Message output alongside parsed structure

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with Details."""
    title:str=Field(..., description="The title of the Movie")
    year:int=Field(..., description="This year the movie was released")
    director:str=Field(..., description="The director of the Movie")
    rating:float=Field(..., description="This is the rating of the movie out of 10")

model_with_structure=model.with_structured_output(Movie, include_raw=True)

In [35]:
response = model_with_structure.invoke("Provide the details of Movie Final Destination")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the details of the movie "Final Destination." I need to use the Movie function to get the information. Let me check the parameters required: title, year, director, and rating. I know the title is "Final Destination." I should confirm the release year, which I think is 2000. The director might be James Wong. As for the rating, maybe around 7.5 out of 10. Let me make sure these details are correct. Once I have all the required parameters, I can structure the function call accordingly.\n', 'tool_calls': [{'id': 'mr6nvwp42', 'function': {'arguments': '{"director":"James Wong","rating":7.5,"title":"Final Destination","year":2000}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 167, 'prompt_tokens': 234, 'total_tokens': 401, 'completion_time': 0.308991756, 'completion_tokens_details': {'reasoning_tokens': 119}, 'prompt_time': 0.009930715, 'pro

### Nested Structure

In [37]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in Million USD")

In [38]:
model_with_structure = model.with_structured_output(MovieDetails)
response=model_with_structure.invoke("Provide the details of the Movie Inception.")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames')], genres=['Action', 'Sci-Fi', 'Thriller'], budget=160.0)

### TypeDict

It is a special class in Python's typing module that allows you to define a dictionary with a fixed structure and type hints for each key. 


It provides type safety for dictionaries, which is very useful when working with JSON-like data

In [42]:
from traitlets.utils import descriptions
from typing_extensions import TypedDict, Annotated


class MovieDict(TypedDict):
    """A movie with Details."""
    title:Annotated[str, ..., "The title of the Movie"]
    year: Annotated[int, ..., "The year in which movie was released"]
    director: Annotated[str, ..., "The director of the movie."]
    rating: Annotated[float, ..., "Rating of the movie out of 10"]

model_with_typeDict = model.with_structured_output(MovieDict)
response=model_with_typeDict.invoke("Please provide the details of the Movie Spiderman")
response

{'director': 'Jon Watts', 'rating': 7.5, 'title': 'Spiderman', 'year': 2021}

In [ ]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in Million USD")

model_with_typeDict = model.with_structured_output(MovieDetails)
response=model_with_typeDict.invoke("Please provide the details of the Movie Spiderman 2 ")
response

{'budget': 200000000,
 'cast': [{'name': 'Tobey Maguire', 'role': 'Peter Parker / Spider-Man'},
  {'name': 'Kirsten Dunst', 'role': 'Mary Jane Watson'},
  {'name': 'Ralph Fiennes', 'role': 'Doctor Otto Octavius / Doctor Octopus'}],
 'genres': ['Action', 'Adventure', 'Superhero'],
 'title': 'Spider-Man 2',
 'year': 2004}

In [47]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### Data Classes

It is a module in Python's library that helps us to create clean, simple classes to store data. 

it is used when required: 
* Simple data container
* Good performance
* Don't need strict validation
* When we are writing a internal data structure
* clean, readable code with minimal boilerplate.


Also Pydantic is used when 
* Strong validation needed
* When we are working with the external data (APIs, JSON, user input)
* When we want clear error messages

In [48]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [57]:
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name:str = Field(description="It's the name of the person.")
    email:str = Field(description="It is the emailId of the person")
    phone: str = Field(description="It is the mobile number of the Person.")



llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Pass the initialized model object instead of a string
agent = create_agent(
    model=llm,
    response_format=ContactInfo
)


result = agent.invoke({
    "messages": [{
        "role":"user",
        "content":"extract contact info from : I'm Ashish Singh, my emailid is the ashish@gmail.com and I have a very beautiful mobile number 919191232."
    }]
})

result

{'messages': [HumanMessage(content="extract contact info from : I'm Ashish Singh, my emailid is the ashish@gmail.com and I have a very beautiful mobile number 919191232.", additional_kwargs={}, response_metadata={}, id='8abad275-e70a-41c3-abf1-4aa7a13a2de7'),
  AIMessage(content='{"name": "Ashish Singh", "email": "ashish@gmail.com", "phone": "919191232"}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e738a-c07d-74c3-b5fb-6cf0d34e043a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 43, 'output_tokens': 179, 'total_tokens': 222, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 146}})],
 'structured_response': ContactInfo(name='Ashish Singh', email='ashish@gmail.com', phone='919191232')}

In [58]:

result["structured_response"]

ContactInfo(name='Ashish Singh', email='ashish@gmail.com', phone='919191232')

In [59]:
## With TypeDict

from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfoTypeDict(TypedDict):
    """Contact information for a person."""
    name:str
    email:str
    phone:str



llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Pass the initialized model object instead of a string
agent = create_agent(
    model=llm,
    response_format=ContactInfoTypeDict
)


result = agent.invoke({
    "messages": [{
        "role":"user",
        "content":"extract contact info from : I'm Ashish Singh, my emailid is the ashish@gmail.com and I have a very beautiful mobile number 919191232."
    }]
})

result

{'messages': [HumanMessage(content="extract contact info from : I'm Ashish Singh, my emailid is the ashish@gmail.com and I have a very beautiful mobile number 919191232.", additional_kwargs={}, response_metadata={}, id='5d0eadbb-a185-42dc-98e2-b8229a954692'),
  AIMessage(content='{\n  "name": "Ashish Singh",\n  "email": "ashish@gmail.com",\n  "phone": "919191232"\n}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e738e-8d1c-7153-9ddd-22d5b6338424-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 43, 'output_tokens': 123, 'total_tokens': 166, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 81}})],
 'structured_response': {'name': 'Ashish Singh',
  'email': 'ashish@gmail.com',
  'phone': '919191232'}}

In [60]:
result["structured_response"]

{'name': 'Ashish Singh', 'email': 'ashish@gmail.com', 'phone': '919191232'}

In [61]:
## With Dataclasses

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass

class ContactInfoUsingDataclass:
    """Contact information of a Person"""
    name: str
    email:str
    phone:str


agent = create_agent(
    model=llm,
    response_format=ContactInfoUsingDataclass
)

result = agent.invoke({
    "messages": [{
        "role":"user",
        "content":"extract contact info from : I'm Ashish Singh, my emailid is the ashish@gmail.com and I have a very beautiful mobile number 919191232."
    }]
})

result

{'messages': [HumanMessage(content="extract contact info from : I'm Ashish Singh, my emailid is the ashish@gmail.com and I have a very beautiful mobile number 919191232.", additional_kwargs={}, response_metadata={}, id='742574f8-811c-42d1-aa15-d047a62d6ab8'),
  AIMessage(content='{\n  "name": "Ashish Singh",\n  "email": "ashish@gmail.com",\n  "phone": "919191232"\n}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7398-05f3-7b61-8f1e-4c887256eb42-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 43, 'output_tokens': 127, 'total_tokens': 170, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 85}})],
 'structured_response': ContactInfoUsingDataclass(name='Ashish Singh', email='ashish@gmail.com', phone='919191232')}

In [62]:
result["structured_response"]

ContactInfoUsingDataclass(name='Ashish Singh', email='ashish@gmail.com', phone='919191232')